In [1]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
import joblib
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("test.csv")

In [3]:
df.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [ ]:
# Filling null values
age_median = df['Age'].median()
df['Age'] = df['Age'].fillna(age_median)

In [ ]:
# Replacing Male -> 0 and Female -> 1
df['Sex'] = df['Sex'].replace({'male': '0' , 'female': '1'})

In [ ]:
# Removing Ticket Column
df.drop('Ticket', axis=1, inplace=True)

In [ ]:
# Rounding up the Fare column data upto 2 decimal places
df['Fare'] = df['Fare'].round(2)

In [8]:
df.dtypes

PassengerId      int64
Pclass           int64
Name            object
Sex             object
Age            float64
SibSp            int64
Parch            int64
Fare           float64
Cabin           object
Embarked        object
dtype: object

In [9]:
# 1. Cabin_Known
df['Cabin_Known'] = df['Cabin'].notna().astype(int)

# 2. Cabin_Deck
df['Cabin_Deck'] = df['Cabin'].fillna('Unknown').str[0]

# 3. One-hot encode Cabin_Deck if used in modeling
cabin_dummies = pd.get_dummies(df['Cabin_Deck'], prefix='Deck')
df = pd.concat([df, cabin_dummies], axis=1)

# 4. Drop the original Cabin column
df.drop('Cabin', axis=1, inplace=True)

In [10]:
df.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Fare,Embarked,Cabin_Known,Cabin_Deck,Deck_A,Deck_B,Deck_C,Deck_D,Deck_E,Deck_F,Deck_G,Deck_U
0,892,3,"Kelly, Mr. James",0,34.5,0,0,7.83,Q,0,U,False,False,False,False,False,False,False,True
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",1,47.0,1,0,7.00,S,0,U,False,False,False,False,False,False,False,True
2,894,2,"Myles, Mr. Thomas Francis",0,62.0,0,0,9.69,Q,0,U,False,False,False,False,False,False,False,True
3,895,3,"Wirz, Mr. Albert",0,27.0,0,0,8.66,S,0,U,False,False,False,False,False,False,False,True
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",1,22.0,1,1,12.29,S,0,U,False,False,False,False,False,False,False,True


In [ ]:
# Finding Missing Values
missing_values = df.isnull().sum()
print(missing_values)

PassengerId    0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Fare           1
Embarked       0
Cabin_Known    0
Cabin_Deck     0
Deck_A         0
Deck_B         0
Deck_C         0
Deck_D         0
Deck_E         0
Deck_F         0
Deck_G         0
Deck_U         0
dtype: int64


In [ ]:
# Adding new columns
embarked_mode = df['Embarked'].mode(
)[0] if 'Embarked' in df.columns and not df['Embarked'].mode().empty else "S"

In [ ]:
# Adding new columns
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

In [14]:
df.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Fare,Embarked,Cabin_Known,...,Deck_A,Deck_B,Deck_C,Deck_D,Deck_E,Deck_F,Deck_G,Deck_U,FamilySize,IsAlone
0,892,3,"Kelly, Mr. James",0,34.5,0,0,7.83,Q,0,...,False,False,False,False,False,False,False,True,1,1
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",1,47.0,1,0,7.00,S,0,...,False,False,False,False,False,False,False,True,2,0
2,894,2,"Myles, Mr. Thomas Francis",0,62.0,0,0,9.69,Q,0,...,False,False,False,False,False,False,False,True,1,1
3,895,3,"Wirz, Mr. Albert",0,27.0,0,0,8.66,S,0,...,False,False,False,False,False,False,False,True,1,1
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",1,22.0,1,1,12.29,S,0,...,False,False,False,False,False,False,False,True,3,0


In [15]:
df.dtypes

PassengerId      int64
Pclass           int64
Name            object
Sex             object
Age            float64
SibSp            int64
Parch            int64
Fare           float64
Embarked        object
Cabin_Known      int64
Cabin_Deck      object
Deck_A            bool
Deck_B            bool
Deck_C            bool
Deck_D            bool
Deck_E            bool
Deck_F            bool
Deck_G            bool
Deck_U            bool
FamilySize       int64
IsAlone          int64
dtype: object

In [18]:
# Define the desired dtypes for each column
dtype_mapping = {
    'PassengerId': 'int64',
    'Pclass':     'int64',
    'Name':       'object',
    'Sex':        'category',
    'Age':        'float64',
    'SibSp':      'int64',
    'Parch':      'int64',
    'Fare':       'float64',
    'Embarked':   'category',
    'Cabin_Known': 'int64',
    'Cabin_Deck': 'category',
    'Deck_A':     'bool',
    'Deck_B':     'bool',
    'Deck_C':     'bool',
    'Deck_D':     'bool',
    'Deck_E':     'bool',
    'Deck_F':     'bool',
    'Deck_G':     'bool',
    'Deck_T':     'bool',
    'Deck_U':     'bool',
    'FamilySize': 'int64',
    'IsAlone':    'int64'
}

# Apply the dtype conversions
for col, dtype in dtype_mapping.items():
    if col in df.columns:
        df[col] = df[col].astype(dtype)

# Verify the changes
print(df.dtypes)

PassengerId       int64
Pclass            int64
Name             object
Sex            category
Age             float64
SibSp             int64
Parch             int64
Fare            float64
Embarked       category
Cabin_Known       int64
Cabin_Deck     category
Deck_A             bool
Deck_B             bool
Deck_C             bool
Deck_D             bool
Deck_E             bool
Deck_F             bool
Deck_G             bool
Deck_U             bool
FamilySize        int64
IsAlone           int64
dtype: object


In [19]:
df.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Fare,Embarked,Cabin_Known,...,Deck_A,Deck_B,Deck_C,Deck_D,Deck_E,Deck_F,Deck_G,Deck_U,FamilySize,IsAlone
0,892,3,"Kelly, Mr. James",0,34.5,0,0,7.83,Q,0,...,False,False,False,False,False,False,False,True,1,1
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",1,47.0,1,0,7.00,S,0,...,False,False,False,False,False,False,False,True,2,0
2,894,2,"Myles, Mr. Thomas Francis",0,62.0,0,0,9.69,Q,0,...,False,False,False,False,False,False,False,True,1,1
3,895,3,"Wirz, Mr. Albert",0,27.0,0,0,8.66,S,0,...,False,False,False,False,False,False,False,True,1,1
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",1,22.0,1,1,12.29,S,0,...,False,False,False,False,False,False,False,True,3,0


In [20]:
df.columns

Index(['PassengerId', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare',
       'Embarked', 'Cabin_Known', 'Cabin_Deck', 'Deck_A', 'Deck_B', 'Deck_C',
       'Deck_D', 'Deck_E', 'Deck_F', 'Deck_G', 'Deck_U', 'FamilySize',
       'IsAlone'],
      dtype='object')

In [ ]:
# Removing Cabin_Deck as not required
df.drop('Cabin_Deck', axis=1, inplace=True)

In [23]:
df.to_csv("test_preprocessed.csv", index=False)